## XGBoostのモデル構築

In [8]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)

In [3]:
x_train = pd.read_csv('../data/treated-data/df_x_train.csv')
y_train = pd.read_csv('../data/treated-data/df_y_train.csv')
x_test = pd.read_csv('../data/treated-data/df_x_test.csv')
y_test = pd.read_csv('../data/treated-data/df_y_test.csv')

In [5]:
#必要なライブラリの読み込み
import xgboost as xgb

dtrain = xgb.DMatrix(x_train, label=y_train)

#Xgboostのパラメータを設定する．ここでは…
#max_depth: 木構造の最大の深さ
#eta: 学習効率
#objective: 学習の目的
#reg: linear(線形回帰)
#reg: logistic(ロジスティック回帰)
#binary: logistic(2値分類)
#multi: softmax(多値分類（2値でも可能）)
#num_class: クラス数（'objective': 'multi:softmax'の時に必要）
xgb_params = {'max_depth': 3,
              'eta': 0.1,
              'objective': 'multi:softmax',
              'num_class': 2}

#学習の実行．繰り返し計算数1000
gbm = xgb.train(xgb_params, dtrain, num_boost_round=1000)
#テストデータもDMatrix形式に
dtest = xgb.DMatrix(x_test)
#予測
pred = gbm.predict(dtest)

#混合行列で精度を確認する
mat = confusion_matrix(y_test, pred)

#pandasで表の形に
df = pd.DataFrame(mat)
print(df)

      0     1
0  3467    45
1   205  1731


In [7]:
from sklearn.metrics import f1_score
print(f1_score(y_test,pred, average="macro"))

0.948925653751632


In [9]:
date_str = datetime.now().strftime('%Y%m%d')
model_filename = f'../models/xgb_gbm_{date_str}.pkl'

# XGBoostのモデルをPickleで保存
with open(model_filename, 'wb') as f:
    pickle.dump(gbm, f)